# Import

In [10]:
import torch
import torch.nn as nn

In [22]:
class Model(nn.Module):
  def __init__(self, nums_feature):
    super().__init__()
    self.linear = nn.Linear(nums_feature, 1) # z= w1x1+w2x2+b
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):
    z =  self.linear(features)
    out = self.sigmoid(z)
    return out


In [24]:
# create dataset
features = torch.rand(20, 3)

# model
model = Model(features.shape[1])
# model.forwarded(features)
model(features)

tensor([[0.6226],
        [0.6471],
        [0.6490],
        [0.6976],
        [0.6955],
        [0.6602],
        [0.6992],
        [0.6618],
        [0.6424],
        [0.6972],
        [0.6322],
        [0.6664],
        [0.6328],
        [0.6371],
        [0.6647],
        [0.6262],
        [0.7091],
        [0.6917],
        [0.6696],
        [0.7051]], grad_fn=<SigmoidBackward0>)

In [25]:
# show model weight
model.linear.weight

Parameter containing:
tensor([[0.0125, 0.0460, 0.4411]], requires_grad=True)

In [26]:
# show model bias
model.linear.bias

Parameter containing:
tensor([0.4383], requires_grad=True)

In [27]:
!pip install torchinfo

In [28]:
from torchinfo import summary
summary(model, input_size=(features.shape[0], features.shape[1]))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [20, 1]                   --
├─Linear: 1-1                            [20, 1]                   4
├─Sigmoid: 1-2                           [20, 1]                   --
Total params: 4
Trainable params: 4
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

# More Complex

In [29]:
class Model2(nn.Module):
  def __init__(self, nums_feature):
    super().__init__()
    self.linear1 = nn.Linear(3, 2)
    self.relu = nn.ReLU()

    self.linear2 = nn.Linear(2, 1)
    self.sigmoid = nn.Sigmoid()


  def forward(self, features):
    out =  self.linear1(features) # 5
    print(out)
    out = self.relu(out) # 6
    out = self.linear2(out) # 10
    out = self.sigmoid(out)

    return out


In [33]:
features2 = torch.rand(20, 3)

# model
model2 = Model2(features2.shape[1])
# model.forwarded(features)
model2(features2)

tensor([[0.5757],
        [0.6407],
        [0.5378],
        [0.5691],
        [0.5915],
        [0.5254],
        [0.5308],
        [0.6006],
        [0.5406],
        [0.5623],
        [0.5501],
        [0.5716],
        [0.5307],
        [0.5256],
        [0.5575],
        [0.5529],
        [0.5764],
        [0.5769],
        [0.5653],
        [0.5467]], grad_fn=<SigmoidBackward0>)

In [34]:
summary(model2, input_size=(features2.shape[0], features2.shape[1]))

Layer (type:depth-idx)                   Output Shape              Param #
Model2                                   [20, 1]                   --
├─Linear: 1-1                            [20, 2]                   8
├─ReLU: 1-2                              [20, 2]                   --
├─Linear: 1-3                            [20, 1]                   3
├─Sigmoid: 1-4                           [20, 1]                   --
Total params: 11
Trainable params: 11
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

# Model with Sequential Container


In [35]:
class Model3(nn.Module):
  def __init__(self, nums_feature):
    super().__init__()
    self.networks = nn.Sequential(
        nn.Linear(3, 2),
        nn.ReLU(),
        nn.Linear(2, 1),
        nn.Sigmoid()
    )


  def forward(self, features):
    out =  self.networks(features)
    return out


In [36]:
features3 = torch.rand(20, 3)

# model
model3 = Model3(features3.shape[1])
# model.forwarded(features)
model3(features3)

tensor([[0.4617],
        [0.4425],
        [0.4617],
        [0.4617],
        [0.4251],
        [0.4504],
        [0.4517],
        [0.4434],
        [0.4365],
        [0.4495],
        [0.4355],
        [0.4424],
        [0.4095],
        [0.4617],
        [0.4545],
        [0.4617],
        [0.4606],
        [0.4538],
        [0.4537],
        [0.4617]], grad_fn=<SigmoidBackward0>)

In [37]:
summary(model3, input_size=(features3.shape[0], features3.shape[1]))

Layer (type:depth-idx)                   Output Shape              Param #
Model3                                   [20, 1]                   --
├─Sequential: 1-1                        [20, 1]                   --
│    └─Linear: 2-1                       [20, 2]                   8
│    └─ReLU: 2-2                         [20, 2]                   --
│    └─Linear: 2-3                       [20, 1]                   3
│    └─Sigmoid: 2-4                      [20, 1]                   --
Total params: 11
Trainable params: 11
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

# Conceptual Session-1 Week-4 Pipline (Improvement)

# Data Loading & Basic Preprocessing

In [38]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

columns = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"
]

df = pd.read_csv(url, names=columns)

print(df.head())
df.shape

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  


(768, 9)

In [39]:

# Separate features and labels
X = df.iloc[:,:-1]
y= df.iloc[:,-1]

# Train test split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling

In [40]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# Numpy to Tensor Conversion
PyTorch works with TENSORS. Not numpy arrays. So we convert.

In [41]:
# Convert to tensors
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()

y_train_tensor = torch.from_numpy(y_train.values).float().view(-1,1)
y_test_tensor = torch.from_numpy(y_test.values).float().view(-1,1)


# Defining the Model

In [46]:
class OurNN(nn.Module):
  def __init__(self, input_size):
    #intialize wight and bias
    super().__init__()
    # self.weights= torch.rand(input_size, 1, requires_grad=True)
    # self.bias= torch.zeros(1, requires_grad=True)
    self.linear = nn.Linear(input_size, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, X):
    #forward pass
    z = self.linear(X)
    out = self.sigmoid(z)
    return out

  def loss_function(self,y_pred, y):
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)
    loss = -(y*torch.log(y_pred) + (1-y)*torch.log(1-y_pred))
    return loss.mean()

# Training Pipeline

In [50]:
#hyperparameters

learning_rate = 0.01
epochs = 5000


# creating model instance
model = OurNN(X_train_tensor.shape[1])

#training loop

for epoch in range(epochs):

    # 1. Forward pass
    y_pred = model(X_train_tensor)

    # 2. Loss calculation
    loss = model.loss_function(y_pred, y_train_tensor)

    # 3. Backward pass
    loss.backward()

    # 4. Update parameters
    with torch.no_grad():
        model.linear.weight -= learning_rate * model.linear.weight.grad
        model.linear.bias -= learning_rate * model.linear.bias.grad

    # 5. Zero gradients
    model.linear.weight.grad.zero_()
    model.linear.bias.grad.zero_()
    print(f"Epoch : {epoch+1} , Loss : {loss.item()}")

Epoch : 1 , Loss : 0.7279122471809387
Epoch : 2 , Loss : 0.7266820073127747
Epoch : 3 , Loss : 0.7254595160484314
Epoch : 4 , Loss : 0.7242446541786194
Epoch : 5 , Loss : 0.7230375409126282
Epoch : 6 , Loss : 0.721838116645813
Epoch : 7 , Loss : 0.7206461429595947
Epoch : 8 , Loss : 0.7194617390632629
Epoch : 9 , Loss : 0.7182847261428833
Epoch : 10 , Loss : 0.7171152234077454
Epoch : 11 , Loss : 0.7159530520439148
Epoch : 12 , Loss : 0.7147981524467468
Epoch : 13 , Loss : 0.7136505842208862
Epoch : 14 , Loss : 0.7125102877616882
Epoch : 15 , Loss : 0.7113771438598633
Epoch : 16 , Loss : 0.7102510333061218
Epoch : 17 , Loss : 0.7091320157051086
Epoch : 18 , Loss : 0.7080201506614685
Epoch : 19 , Loss : 0.7069151401519775
Epoch : 20 , Loss : 0.7058171629905701
Epoch : 21 , Loss : 0.7047260999679565
Epoch : 22 , Loss : 0.7036418318748474
Epoch : 23 , Loss : 0.7025644779205322
Epoch : 24 , Loss : 0.7014937400817871
Epoch : 25 , Loss : 0.7004297375679016
Epoch : 26 , Loss : 0.6993724107742

# Model Evaluation

In [51]:
with torch.no_grad():
    y_pred = model(X_test_tensor)
    y_pred_class = (y_pred>0.5 ).float()
    accuracy = (y_pred_class == y_test_tensor).float().mean()

    print(f"Accuracy : {accuracy.item()}")

Accuracy : 0.7532467246055603
